# Add the pinned Win64 feature data

- Download feature JSONL archives, not executable binaries.
- Verify archive hashes before extraction; reuse verified files on rerun.
- Keep official train/test boundaries. The analysis notebook audits duplicates.

## 1. Check sources and storage

Read the pinned revision and checksums. Store temporary downloads inside the project.

In [1]:
# Load project paths
# Keep all acquisition files inside the project.

from pathlib import Path
import os, json, hashlib, shutil, zipfile, time
from multi_label_static_analysis.config import project_paths

P = project_paths()
P.data.mkdir(parents=True, exist_ok=True)


In [2]:
# Configure the download cache
# Set cache locations before importing the Hub client.

SOURCE = json.loads((P.project / "configs/ember2024/win64_sources.json").read_text())
DOWNLOAD = P.artifacts / "downloads"
DOWNLOAD.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(DOWNLOAD / "hf_home")
os.environ["HF_XET_CACHE"] = str(DOWNLOAD / "xet")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import hf_hub_download


/Users/satvik/Desktop/projects/multi-label-static-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Check source revision and capacity
# Validate the pinned source and disk capacity before continuing.

from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()
print("Pinned dataset revision:", SOURCE["revision"])
print("Archives GB:", sum(f["bytes"] for f in SOURCE["files"]) / 1e9)
assert (
    shutil.disk_usage(P.root).free > 35 * 10**9
), "Need at least 35 GB free for archives, features, and expanded model inputs."
print("PASS: pinned Win64 sources and disk capacity checked.")


Pinned dataset revision: 3d23efef7c0f0b702c5024400cfff4c3744a3832
Archives GB: 7.029602617
PASS: pinned Win64 sources and disk capacity checked.


## 2. Reuse or extract verified archives

Process one archive at a time. Hash it before extraction, record each JSONL file, then remove the compressed copy.

In [ ]:
# Open the completion record
# A completed record lets reruns avoid network transfer.

from multi_label_static_analysis.data_acquisition import process_archive

completed_path = P.artifacts / "win64_download_manifest.json"
completed = (
    json.loads(completed_path.read_text())
    if completed_path.exists()
    else {"revision": SOURCE["revision"], "archives": {}}
)
assert completed["revision"] == SOURCE["revision"]


In [ ]:
# Process the pinned archives
# The helper verifies hashes and extracts JSONL members using safe base filenames.

for item in sorted(SOURCE["files"], key=lambda f: "_train" not in f["filename"]):
    process_archive(item, P, SOURCE, DOWNLOAD, completed, completed_path)
print(
    "PASS: both archives verified and JSONL feature files extracted; completion manifest saved."
)


## 3. Verify the extracted data

Check schema and expected raw-row coverage. Unique-record counts are established later by the analysis audit.

In [ ]:
# Check the feature files
# Repeated raw rows are allowed here; the analysis step handles deduplication.

for name, minimum in [("Win64_train.zip", 520000), ("Win64_test.zip", 120000)]:
    info = completed["archives"][name]
    rows = sum(entry["raw_rows"] for entry in info["extracted"])
    assert rows >= minimum
    for entry in info["extracted"]:
        path = P.data / entry["file"]
        assert path.stat().st_size == entry["bytes"]
        with path.open() as stream:
            example = json.loads(stream.readline())
        assert example["file_type"] == "Win64"
    print(name, "verified JSONL files:", len(info["extracted"]), "raw rows:", rows)
print(
    "PASS: Win64 train/test feature schema and expected row-count coverage checked; deduplication follows in research audit."
)


## Execution note

The setup cells were checked locally. Transfer cells are intentionally unexecuted: the local completion manifest is missing, and this review does not re-download archives. Extraction and reuse logic were checked with offline fixtures. Run acquisition explicitly when you intend to restore the pinned source data.